In [8]:
import numpy as np
import pandas as pd

This cell imports the necessary libraries:
- `numpy` as `np` for numerical operations, especially for handling arrays and matrices.
- `pandas` as `pd` for data manipulation and analysis, primarily for working with DataFrames.

In [9]:
df = pd.DataFrame([[8,8,4],[7,9,5],[6,10,6],[5,12,7]],columns=['cgpa','profile_score','lpa'])

This cell creates a pandas DataFrame named `df` with sample data. It represents a small dataset with columns 'cgpa', 'profile_score', and 'lpa'. This data will be used to train and test the simple neural network.

In [10]:
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


This cell simply displays the `df` DataFrame to show its contents.

In [11]:
def initialize_parameters(layer_dims):
  np.random.seed(3)
  parameters = {}
  L = len(layer_dims)

  for l in range(1,L):
    parameters['W'+str(l)] = np.ones((layer_dims[l-1],layer_dims[l]))*0.1
    parameters['b'+str(l)] = np.zeros((layer_dims[l],1))

  return parameters

This function `initialize_parameters` is designed to set up the initial weights (W) and biases (b) for a neural network with a given `layer_dims` architecture.

- It initializes weights with small positive values (0.1) and biases with zeros.
- `np.random.seed(3)` ensures reproducibility of random initializations.
- The loop iterates through each layer (starting from the first hidden layer) and creates weight matrices `Wl` and bias vectors `bl`.
- `Wl` is initialized as a matrix with dimensions `(layer_dims[l-1], layer_dims[l])`.
- `bl` is initialized as a column vector with dimensions `(layer_dims[l], 1)`.

In [12]:
initialize_parameters([2,2,1])

{'W1': array([[0.1, 0.1],
        [0.1, 0.1]]),
 'b1': array([[0.],
        [0.]]),
 'W2': array([[0.1],
        [0.1]]),
 'b2': array([[0.]])}

This cell calls the `initialize_parameters` function to create the initial weights and biases for a network with an architecture of `[2, 2, 1]`.

- The input layer has 2 features.
- The hidden layer has 2 neurons.
- The output layer has 1 neuron.

The `parameters` dictionary will store the initialized `W1`, `b1`, `W2`, and `b2`.

In [13]:
def linear_forward(A_prev,W,b):
  Z = np.dot(W.T, A_prev) + b
  return Z

The `linear_forward` function performs the linear part of a neural network's forward propagation for a single layer.

- It takes the activation from the previous layer (`A_prev`), the weights (`W`), and the biases (`b`) of the current layer.
- It calculates `Z = W.T * A_prev + b`, where `W.T` is the transpose of the weight matrix.
- `Z` is the input to the activation function for the current layer.

In [14]:
def L_layer_forward(X,parameters):

  A = X
  L = len(parameters)//2

  for l in range(1,L+1):
    A_prev = A
    Wl = parameters['W'+str(l)]
    bl = parameters['b'+str(l)]
    print("A"+str(l-1)+": ",A_prev)
    print("W"+str(l)+": ",Wl)
    print("b"+str(l)+": ",bl)
    print("--"*20)

    A = linear_forward(A_prev,Wl,bl)
    print("A"+str(l)+": ",A)
    print("--"*20)
  return A,A_prev

The `L_layer_forward` function implements the full forward propagation for an L-layer neural network.

- It takes the input `X` and the `parameters` (weights and biases) of the network.
- It iterates through each layer, performing the `linear_forward` calculation.
- It prints the intermediate values of `A_prev` (activations from the previous layer), `Wl` (weights for the current layer), and `bl` (biases for the current layer) for debugging or observation.
- For this simple network, there are no explicit activation functions (like ReLU or Sigmoid) between layers, so `A` simply becomes `Z` from `linear_forward`.
- It returns the final output `A` (which is `y_hat`) and the activation of the last hidden layer (`A_prev`).

In [15]:
X = df[['cgpa','profile_score']].values[0].reshape(2,1)
y = df[['lpa']].values[0][0]

parameters = initialize_parameters([2,2,1])
y_hat,A1 = L_layer_forward(X,parameters)

A0:  [[8]
 [8]]
W1:  [[0.1 0.1]
 [0.1 0.1]]
b1:  [[0.]
 [0.]]
----------------------------------------
A1:  [[1.6]
 [1.6]]
----------------------------------------
A1:  [[1.6]
 [1.6]]
W2:  [[0.1]
 [0.1]]
b2:  [[0.]]
----------------------------------------
A2:  [[0.32]]
----------------------------------------


This cell demonstrates a single forward pass for the first data point in the `df` DataFrame.

- `X` is extracted as the input features (cgpa, profile_score) for the first row and reshaped into a column vector.
- `y` is extracted as the true label (lpa) for the first row.
- `parameters` are re-initialized for a fresh start.
- `L_layer_forward` is called to get the predicted output `y_hat` and the activation of the hidden layer `A1`.

In [16]:
y_hat = y_hat[0][0]

This line extracts the scalar prediction from the `y_hat` array, as `y_hat` is initially an array containing a single value.

In [17]:
A1

array([[1.6],
       [1.6]])

This cell displays the activation of the first hidden layer (`A1`), which is the output of the `linear_forward` step for the first layer in the previous execution.

In [19]:
def update_parameters(parameters,y,y_hat,A1,X):
  parameters['W2'][0][0] = parameters['W2'][0][0] + (0.001*2*(y-y_hat)*A1[0][0])
  parameters['W2'][1][0] = parameters['W2'][1][0] + (0.001*2*(y-y_hat)*A1[1][0])
  parameters['b2'][0][0] = parameters['b2'][0][0] + (0.001*2*(y-y_hat))

  parameters['W1'][0][0] = parameters['W1'][0][0] + (0.001*2*(y-y_hat)*parameters['W2'][0][0]*X[0][0])
  parameters['W1'][0][1] = parameters['W1'][0][1] + (0.001*2*(y-y_hat)*parameters['W2'][0][0]*X[1][0])
  parameters['b1'][0][0] = parameters['b1'][0][0] + (0.001*2*(y-y_hat)*parameters['W2'][0][0])

  parameters['W1'][1][0] = parameters['W1'][1][0] + (0.001*2*(y-y_hat)*parameters['W2'][1][0]*X[0][0])
  parameters['W1'][1][1] = parameters['W1'][1][1] + (0.001*2*(y-y_hat)*parameters['W2'][1][0]*X[1][0])
  parameters['b1'][1][0] = parameters['b1'][1][0] + (0.001*2*(y-y_hat)*parameters['W2'][1][0])

  return parameters

The `update_parameters` function performs a manual, simplified backpropagation and parameter update for this specific 2-layer neural network.

- It calculates the gradients for `W2`, `b2`, `W1`, and `b1` using the squared error loss `(y-y_hat)` and the activations `A1` and input `X`.
- A learning rate of `0.001` is used to adjust the parameters.
- This is a direct, hardcoded implementation of gradient descent steps for this particular network architecture.

In [20]:
update_parameters(parameters,y,y_hat,A1,X)

{'W1': array([[0.10658137, 0.10658137],
        [0.10658137, 0.10658137]]),
 'b1': array([[0.00082267],
        [0.00082267]]),
 'W2': array([[0.111776],
        [0.111776]]),
 'b2': array([[0.00736]])}

This cell calls the `update_parameters` function, applying the calculated gradients to modify the network's weights and biases based on the error from the first training example. The updated `parameters` dictionary is then displayed.

In [21]:
parameters

{'W1': array([[0.10658137, 0.10658137],
        [0.10658137, 0.10658137]]),
 'b1': array([[0.00082267],
        [0.00082267]]),
 'W2': array([[0.111776],
        [0.111776]]),
 'b2': array([[0.00736]])}

In [22]:
X = df[['cgpa','profile_score']].values[1].reshape(2,1)
y = df[['lpa']].values[1][0]


y_hat,A1 = L_layer_forward(X,parameters)
y_hat = y_hat[0][0]
update_parameters(parameters,y,y_hat,A1,X)
parameters

A0:  [[7]
 [9]]
W1:  [[0.10658137 0.10658137]
 [0.10658137 0.10658137]]
b1:  [[0.00082267]
 [0.00082267]]
----------------------------------------
A1:  [[1.70612461]
 [1.70612461]]
----------------------------------------
A1:  [[1.70612461]
 [1.70612461]]
W2:  [[0.111776]
 [0.111776]]
b2:  [[0.00736]]
----------------------------------------
A2:  [[0.38876757]]
----------------------------------------


{'W1': array([[0.11481311, 0.11716504],
        [0.11481311, 0.11716504]]),
 'b1': array([[0.00199863],
        [0.00199863]]),
 'W2': array([[0.12751067],
        [0.12751067]]),
 'b2': array([[0.01658246]])}

In [23]:
X = df[['cgpa','profile_score']].values[2].reshape(2,1)
y = df[['lpa']].values[2][0]


y_hat,A1 = L_layer_forward(X,parameters)
y_hat = y_hat[0][0]
update_parameters(parameters,y,y_hat,A1,X)
parameters

A0:  [[ 6]
 [10]]
W1:  [[0.11481311 0.11716504]
 [0.11481311 0.11716504]]
b1:  [[0.00199863]
 [0.00199863]]
----------------------------------------
A1:  [[1.83900839]
 [1.8766392 ]]
----------------------------------------
A1:  [[1.83900839]
 [1.8766392 ]]
W2:  [[0.12751067]
 [0.12751067]]
b2:  [[0.01658246]]
----------------------------------------
A2:  [[0.49036719]]
----------------------------------------


{'W1': array([[0.12458335, 0.13344878],
        [0.12461077, 0.13349447]]),
 'b1': array([[0.00362701],
        [0.00363158]]),
 'W2': array([[0.1477752 ],
        [0.14818986]]),
 'b2': array([[0.02760173]])}

In [24]:
X = df[['cgpa','profile_score']].values[3].reshape(2,1)
y = df[['lpa']].values[3][0]


y_hat,A1 = L_layer_forward(X,parameters)
y_hat = y_hat[0][0]
update_parameters(parameters,y,y_hat,A1,X)
parameters

A0:  [[ 5]
 [12]]
W1:  [[0.12458335 0.13344878]
 [0.12461077 0.13349447]]
b1:  [[0.00362701]
 [0.00363158]]
----------------------------------------
A1:  [[2.12187303]
 [2.2728091 ]]
----------------------------------------
A1:  [[2.12187303]
 [2.2728091 ]]
W2:  [[0.1477752 ]
 [0.14818986]]
b2:  [[0.02760173]]
----------------------------------------
A2:  [[0.6779692]]
----------------------------------------


{'W1': array([[0.13562189, 0.15994127],
        [0.13579618, 0.16033944]]),
 'b1': array([[0.00583472],
        [0.00586866]]),
 'W2': array([[0.17460429],
        [0.1769274 ]]),
 'b2': array([[0.04024579]])}

In [25]:
# epochs implementation

parameters = initialize_parameters([2,2,1])
epochs = 5

for i in range(epochs):

  Loss = []

  for j in range(df.shape[0]):

    X = df[['cgpa','profile_score']].values[j].reshape(2,1)
    y = df[['lpa']].values[j][0]

    y_hat,A1 = L_layer_forward(X,parameters)
    y_hat = y_hat[0][0]

    update_parameters(parameters,y,y_hat,A1,X)
    Loss.append((y-y_hat)**2)

  print('Epoch - ',i+1,'Loss - ',np.array(Loss).mean())

parameters


A0:  [[8]
 [8]]
W1:  [[0.1 0.1]
 [0.1 0.1]]
b1:  [[0.]
 [0.]]
----------------------------------------
A1:  [[1.6]
 [1.6]]
----------------------------------------
A1:  [[1.6]
 [1.6]]
W2:  [[0.1]
 [0.1]]
b2:  [[0.]]
----------------------------------------
A2:  [[0.32]]
----------------------------------------
A0:  [[7]
 [9]]
W1:  [[0.10658137 0.10658137]
 [0.10658137 0.10658137]]
b1:  [[0.00082267]
 [0.00082267]]
----------------------------------------
A1:  [[1.70612461]
 [1.70612461]]
----------------------------------------
A1:  [[1.70612461]
 [1.70612461]]
W2:  [[0.111776]
 [0.111776]]
b2:  [[0.00736]]
----------------------------------------
A2:  [[0.38876757]]
----------------------------------------
A0:  [[ 6]
 [10]]
W1:  [[0.11481311 0.11716504]
 [0.11481311 0.11716504]]
b1:  [[0.00199863]
 [0.00199863]]
----------------------------------------
A1:  [[1.83900839]
 [1.8766392 ]]
----------------------------------------
A1:  [[1.83900839]
 [1.8766392 ]]
W2:  [[0.12751067]
 [0.12

{'W1': array([[0.273603  , 0.3993222 ],
        [0.28787155, 0.42586102]]),
 'b1': array([[0.02885522],
        [0.03133223]]),
 'W2': array([[0.42574893],
        [0.50219328]]),
 'b2': array([[0.11841278]])}

This cell implements the full training loop for the neural network over multiple `epochs`.

- **`parameters = initialize_parameters([2,2,1])`**: Re-initializes the weights and biases for a fresh training run.
- **`epochs = 5`**: Sets the number of times the entire dataset will be passed through the network.
- **Outer Loop (`for i in range(epochs):`)**: Iterates for each epoch.
  - **`Loss = []`**: Initializes a list to store the squared error for each example within the current epoch.
  - **Inner Loop (`for j in range(df.shape[0]):`)**: Iterates through each training example in the DataFrame.
    - **`X` and `y` extraction**: Extracts the input features and true label for the current example.
    - **`y_hat, A1 = L_layer_forward(X, parameters)`**: Performs the forward pass to get the prediction and hidden layer activation.
    - **`y_hat = y_hat[0][0]`**: Extracts the scalar prediction.
    - **`update_parameters(parameters, y, y_hat, A1, X)`**: Updates the network's weights and biases based on the current example's error.
    - **`Loss.append((y-y_hat)**2)`**: Calculates and stores the squared error for the current example.
  - **`print('Epoch - ', i+1, 'Loss - ', np.array(Loss).mean())`**: After processing all examples in an epoch, it prints the average loss for that epoch.
- Finally, `parameters` are printed to show the learned weights and biases after all epochs are completed.

### Mathematical Explanation of the Training Process

This section details the mathematical operations performed during the forward pass, loss calculation, and parameter updates within the provided training loop.

#### 1. Forward Propagation (`L_layer_forward` and `linear_forward`)

The network consists of two layers: an input layer, a hidden layer, and an output layer. The `initialize_parameters([2,2,1])` indicates:
- Input features: $N_0 = 2$
- Hidden layer neurons: $N_1 = 2$
- Output layer neurons: $N_2 = 1$

**For the first layer (Input to Hidden Layer):**

Given:
- Input features $X$ (denoted as $A^{[0]}$) of shape $(N_0, 1)$ or $(2, 1)$.
- Weights $W^{[1]}$ of shape $(N_0, N_1)$ or $(2, 2)$.
- Biases $b^{[1]}$ of shape $(N_1, 1)$ or $(2, 1)$.

The linear transformation for the hidden layer is calculated as:
$Z^{[1]} = (W^{[1]})^T A^{[0]} + b^{[1]}$

Where:
- $(W^{[1]})^T$ is the transpose of the weight matrix $W^{[1]}$, with shape $(N_1, N_0)$ or $(2, 2)$.
- $A^{[0]}$ is the input vector $X$, with shape $(N_0, 1)$ or $(2, 1)$.
- The result $(W^{[1]})^T A^{[0]}$ has shape $(N_1, 1)$ or $(2, 1)$.
- $b^{[1]}$ is added element-wise.

In the provided code, there's no explicit activation function after $Z^{[1]}$, so $A^{[1]} = Z^{[1]}$.

**For the second layer (Hidden Layer to Output Layer):**

Given:
- Activations from the hidden layer $A^{[1]}$ of shape $(N_1, 1)$ or $(2, 1)$.
- Weights $W^{[2]}$ of shape $(N_1, N_2)$ or $(2, 1)$.
- Biases $b^{[2]}$ of shape $(N_2, 1)$ or $(1, 1)$.

The linear transformation for the output layer is calculated as:
$Z^{[2]} = (W^{[2]})^T A^{[1]} + b^{[2]}$

Where:
- $(W^{[2]})^T$ is the transpose of the weight matrix $W^{[2]}$, with shape $(N_2, N_1)$ or $(1, 2)$.
- $A^{[1]}$ has shape $(N_1, 1)$ or $(2, 1)$.
- The result $(W^{[2]})^T A^{[1]}$ has shape $(N_2, 1)$ or $(1, 1)$.
- $b^{[2]}$ is added element-wise.

The final prediction, $y_{hat}$, is $A^{[2]} = Z^{[2]}$. The scalar prediction $y_{hat}$ is then extracted as $y_{hat} = A^{[2]}[0][0]$.

#### 2. Loss Function

The code uses the **Mean Squared Error (MSE)** as the loss function, but it calculates the squared error for each individual training example and averages them at the end of an epoch. For a single example, the squared error is:

$L = (y - y_{hat})^2$

Where:
- $y$ is the true label.
- $y_{hat}$ is the predicted label.

#### 3. Parameter Update (`update_parameters`)

The parameters are updated using a basic form of gradient descent. The learning rate is fixed at $\alpha = 0.001$. The update rules are derived from the gradients of the loss function with respect to each parameter. Since $L = (y - y_{hat})^2$, the gradient for a parameter $\theta$ is $\frac{\partial L}{\partial \theta} = 2(y - y_{hat}) \frac{\partial (-y_{hat})}{\partial \theta}$.

Let's denote $\delta = 2(y - y_{hat})$. The updates are:

**For the Output Layer (Layer 2):**

- **Bias $b^{[2]}$:**
  $b^{[2]}_{new} = b^{[2]}_{old} + \alpha \frac{\partial L}{\partial b^{[2]}} = b^{[2]}_{old} + \alpha \delta \cdot 1$
  Specifically, `parameters['b2'][0][0] = parameters['b2'][0][0] + (0.001*2*(y-y_hat))`

- **Weights $W^{[2]}$:**
  $W^{[2]}_{new} = W^{[2]}_{old} + \alpha \frac{\partial L}{\partial W^{[2]}} = W^{[2]}_{old} + \alpha \delta \cdot A^{[1]}$
  Specifically, for $W^{[2]}_{0,0}$ and $W^{[2]}_{1,0}$ (assuming $W^{[2]}$ is column vector of shape $(2,1)$ as per initialization):
  `parameters['W2'][0][0] = parameters['W2'][0][0] + (0.001*2*(y-y_hat)*A1[0][0])`
  `parameters['W2'][1][0] = parameters['W2'][1][0] + (0.001*2*(y-y_hat)*A1[1][0])`

**For the Hidden Layer (Layer 1):**

The gradients for the first layer depend on the gradients propagated back from the second layer. Using the chain rule:

- **Bias $b^{[1]}$:**
  $b^{[1]}_{new} = b^{[1]}_{old} + \alpha \frac{\partial L}{\partial b^{[1]}} = b^{[1]}_{old} + \alpha \delta \cdot W^{[2]}$ (element-wise multiplication with $W^{[2]}$'s elements)
  Specifically:
  `parameters['b1'][0][0] = parameters['b1'][0][0] + (0.001*2*(y-y_hat)*parameters['W2'][0][0])`
  `parameters['b1'][1][0] = parameters['b1'][1][0] + (0.001*2*(y-y_hat)*parameters['W2'][1][0])`

- **Weights $W^{[1]}$:**
  $W^{[1]}_{new} = W^{[1]}_{old} + \alpha \frac{\partial L}{\partial W^{[1]}} = W^{[1]}_{old} + \alpha \delta \cdot W^{[2]} \cdot X^T$
  Specifically, for each element of $W^{[1]}$ of shape $(2,2)$:
  `parameters['W1'][0][0] = parameters['W1'][0][0] + (0.001*2*(y-y_hat)*parameters['W2'][0][0]*X[0][0])`
  `parameters['W1'][0][1] = parameters['W1'][0][1] + (0.001*2*(y-y_hat)*parameters['W2'][0][0]*X[1][0])`
  `parameters['W1'][1][0] = parameters['W1'][1][0] + (0.001*2*(y-y_hat)*parameters['W2'][1][0]*X[0][0])`
  `parameters['W1'][1][1] = parameters['W1'][1][1] + (0.001*2*(y-y_hat)*parameters['W2'][1][0]*X[1][0])`

This manually implemented `update_parameters` function performs these specific gradient descent steps based on the chosen squared error loss and the network's architecture.